# TAMER OCR - Curriculum Training
This notebook authenticates with Hugging Face, pulls the latest model weights, and trains progressively across 3 difficulty stages.

**Note:** Ensure you are running this in an interactive session so you can paste your token.

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login

print("=========================================")
print("🤗 HUGGING FACE AUTHENTICATION")
print("=========================================")
print("Paste your Hugging Face Token (must have WRITE access).")
hf_token = getpass("HF Token: ").strip()
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=True)
print("✅ Authenticated successfully!")

In [ ]:
# IMPORTANT: Change 'YOUR_GITHUB_USERNAME' below to point to your repository
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/AI_PROJECT_TAMER_Complete.git"

!git clone {REPO_URL}
%cd AI_PROJECT_TAMER_Complete/tamer_ocr

!pip install -r requirements.txt
!pip install datasets huggingface_hub

In [ ]:
import os
from huggingface_hub import hf_hub_download

# This pulls your latest checkpoint so you can resume training after disconnects.
MODEL_REPO = "JJKK1212/tamer-math-ocr" 

os.makedirs('./checkpoints', exist_ok=True)
print("🔄 Checking for existing checkpoints on Hugging Face...")
try:
    hf_hub_download(
        repo_id=MODEL_REPO, 
        filename="latest.pt", 
        local_dir="./checkpoints", 
        repo_type="model", 
        token=os.environ['HF_TOKEN']
    )
    print("✅ Found existing checkpoint. Training will resume from latest.pt!")
except Exception as e:
    print("ℹ️ No existing checkpoint found. Training will start from scratch (Epoch 0).")

### Stage 1: Printed Formulas (`im2latex-100k`)
The model learns strict LaTeX syntax, spatial coverage, and base tree-guided structure.

In [ ]:
# Target: Train up to Epoch 15
!python train.py --datasets im2latex-100k --batch-size 16 --num-epochs 15 --save-every 2

### Stage 2: Clean Handwritten (`mathwriting`)
The model adapts its visual encoder to handle clean, human-written strokes while utilizing the syntax learned in Stage 1.

In [ ]:
# Target: Resume from Epoch 15, Train up to Epoch 30
!python train.py --datasets mathwriting --batch-size 16 --num-epochs 30 --save-every 2

### Stage 3: Messy Handwritten (`crohme` & `hme100k`)
The final stage pushes the model to generalize against noisy backgrounds, crossing-outs, and highly unstructured real-world data.

In [ ]:
# Target: Resume from Epoch 30, Train up to Epoch 50
!python train.py --datasets crohme hme100k --batch-size 16 --num-epochs 50 --save-every 2